# SBE37 operational pipeline

Use this notebook to run the SBE37 workflow end-to-end: `proc_1` reads the deployment, determines the in-water period, trims to that period, and shows a visual confirmation plot; `proc_2` takes the trimmed `proc_1` file, allows manual QC flagging of erroneous or suspicious data, and publishes the final QC output.

**Operator choices**
- Run the full pipeline by leaving all step toggles enabled.
- Run `proc_1` only to regenerate the trimmed file and inspect the interactive review plot.
- Edit `manual_qc_flags` after `proc_1` to define the QC edits that will be applied in `proc_2`.
- Run `proc_2` using the trimmed `proc_1` output and then review the QC output before finalizing.


## Imports

This cell prepares the notebook path and imports the workflow entry points used in the later steps.


In [ ]:
import os
import sys
from pathlib import Path

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'tools').exists() and (NOTEBOOK_DIR / 'mooring_proc' / 'tools').exists():
    sys.path.insert(0, str((NOTEBOOK_DIR / 'mooring_proc').resolve()))

from tools.database_lookup import list_instruments
from tools.helpers import plot_data_by_qc, plot_pressure_comparison
from tools.workflows.run_imos_delivery import run_imos_delivery
from tools.workflows.run_proc1 import run_proc1
from tools.workflows.run_proc2 import run_proc2

## User options

Set the deployment identifier, optional metadata preview filters, and step toggles in this single configuration cell before running any workflow step.


In [ ]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash")

# Optional metadata CSV path. Set this to an absolute path before running preview or workflow steps.
metadata_csv = None

# Required SBE37 deployment identifier from the metadata table.
inst_deploy_id = ""

# Keep this notebook scoped to SBE37 instruments.
instrument = "SBE37"

# Optional metadata preview filters.
show_year = None
show_location = None

# Preview and execution toggles.
show_table_preview = True
run_proc1_step = True
run_proc2_step = True
run_delivery_step = True

# Optional proc_1 review plot controls.
# Expected variables for SBE37: TEMP, CNDC, PSAL, PRES
proc1_plot_variables = None
proc1_plot_flags = None
proc1_zoom_to_good = True

# Manual QC flags will be set after reviewing proc_1 output below.
manual_qc_flags = []

This cell turns the options above into the shared workflow configuration and initializes output tracking for the rest of the notebook.


In [ ]:
def build_workflow_config():
    return {
        "metadata_csv": metadata_csv,
        "inst_deploy_ID": inst_deploy_id,
        "instrument": instrument,
        "manual_qc_flags": manual_qc_flags,
        "reuse_existing_proc1": reuse_existing_proc1,
        "proc_1_input_dataset": proc1_existing_dataset,
    }


def require_ready_config(require_metadata=True, require_instrument=True):
    if require_metadata and not metadata_csv:
        raise ValueError("Set metadata_csv to an absolute metadata CSV path before running this cell.")
    if require_instrument and not str(inst_deploy_id).strip():
        raise ValueError("Set inst_deploy_id to a valid SBE37 deployment identifier before running this cell.")
    return build_workflow_config()


proc1_result = None
proc2_result = None
delivery_result = None
summary = {"proc1": None, "proc2": None, "qc_log": None, "fv00": None, "fv01": None}


## Optional metadata preview

If enabled, this cell lists SBE37 deployments from the metadata table so you can confirm the correct `inst_deploy_id` before processing.


In [ ]:
preview_table = None
if show_table_preview:
    require_ready_config(require_metadata=True, require_instrument=False)
    preview_table = list_instruments(
        metadata_csv,
        year=show_year,
        location=show_location,
        instrument=instrument,
    )
    if preview_table.empty:
        print("No SBE37 deployments matched the preview filters. Clear show_year/show_location or check metadata_csv.")
    else:
        display(preview_table)
else:
    print("Metadata preview skipped.")


## Run `proc_1`

This step reads the SBE37 deployment from metadata, applies deployment-window QC to a review dataset, shows the interactive QC review plot inline, and writes the trimmed proc_1 output.

> The interactive review uses QC-flag plotting from the new-suite helper layer, so you can inspect good and flagged samples without opening saved PNG files.
> Set `reuse_existing_proc1 = True` (and optionally `proc1_existing_dataset`) to reuse an existing proc_1 NetCDF instead of rerunning parse/trim for this step.


In [ ]:
if run_proc1_step:
    workflow_config = require_ready_config()
    proc1_result = run_proc1(workflow_config)
    summary['proc1'] = proc1_result['output_path']
    if proc1_result.get('reused_existing'):
        print(f"proc_1 reused from existing NetCDF: {proc1_result['output_path']}")
    else:
        print(f"proc_1 regenerated: {proc1_result['output_path']}")
    review_figure = plot_data_by_qc(
        proc1_result["review_dataset"],
        variables=proc1_plot_variables,
        flags_to_plot=proc1_plot_flags,
        y_zoom_to_good=proc1_zoom_to_good,
        title=f"SBE37 proc_1 review by QC flag: {inst_deploy_id}",
    )
    review_figure.show()
else:
    print('proc_1 skipped.')


## Optional pressure comparison review

Set `pressure_review_deploy_id` to the `inst_deploy_ID` of a co-deployed reference pressure instrument (e.g. a tide gauge or BPR). The cell overlays the two pressure series and displays residuals inline. Leave both toggles as `None` to skip this step.


In [ ]:
# Optional pressure comparison review
# Set pressure_review_deploy_id to the inst_deploy_ID of a co-deployed reference instrument
# (e.g. a tide gauge or BPR) to generate a pressure-comparison plot after proc_1.
# Leave as None to skip.
pressure_review_deploy_id = None
pressure_review_file = None  # Alternative: explicit path to reference file

if run_proc1_step and proc1_result is not None and (pressure_review_deploy_id or pressure_review_file):
    pressure_fig, _, _ = plot_pressure_comparison(
        df=proc1_result["dataframe"],
        database=metadata_csv,
        pressure_inst_deploy_id=pressure_review_deploy_id,
        pressure_file=pressure_review_file,
        primary_label="SBE37",
    )
    pressure_fig.show()
else:
    print("Pressure comparison skipped (set pressure_review_deploy_id or pressure_review_file to enable).")


## Run `proc_2`

This step applies `manual_qc_flags` to the already-trimmed `proc_1` output and writes the QC log used to document operator edits.

> `proc_2` uses already-trimmed `proc_1` data and does not trim again.


In [ ]:
if run_proc2_step:
    workflow_config = require_ready_config()
    proc2_input = proc1_result["output_path"] if proc1_result else None
    proc2_result = run_proc2(workflow_config, input_dataset=proc2_input)
    summary['proc2'] = proc2_result['output_path']
    summary['qc_log'] = proc2_result['manual_qc_log']
    print(f"proc_2 file: {proc2_result['output_path']}")
    print(f"QC log: {proc2_result['manual_qc_log']}")
else:
    print('proc_2 skipped.')


## Run `imos_delivery`

This step publishes FV00 from `proc_1` and FV01 from `proc_2`, using the files already recorded in metadata when earlier steps are skipped.


In [ ]:
if run_delivery_step:
    workflow_config = require_ready_config()
    delivery_result = run_imos_delivery(workflow_config)
    summary['fv00'] = delivery_result['proc_1_delivery']
    summary['fv01'] = delivery_result['proc_2_delivery']
    print(f"FV00 path: {delivery_result['proc_1_delivery']}")
    print(f"FV01 path: {delivery_result['proc_2_delivery']}")
else:
    print('IMOS delivery skipped.')


## Troubleshooting

- **Missing `inst_deploy_id`**: use the metadata preview cell to find the SBE37 deployment identifier, then rerun the config and workflow cells.
- **Missing `proc_1` file when running `proc_2`**: run `proc_1` first, or confirm the metadata row already points to a valid `proc_1_file` in `proc_1_path`.
- **Empty metadata preview filters**: clear `show_year` and `show_location`, or confirm the metadata CSV contains SBE37 rows for those filters.
- **No plot traces appear**: set `proc1_plot_variables = None` to auto-detect supported SBE37 variables, or confirm the dataset contains matching `*_quality_control` variables.


## Execution summary

This final cell prints the key output paths collected from the steps you ran in this session.


In [ ]:
print(summary)
